# 1. Data inspection

## Objective

This notebook verifies the structure and quality of the SHL grammar-scoring data before modeling. We inspect the train/test tables, target distribution, file availability, duplicate and overlap behavior, and representative audio metadata.

These checks matter because the model depends on aligned filenames, valid audio files, reliable labels, and a target distribution that can be evaluated fairly. They also identify data issues that could otherwise become silent preprocessing or evaluation errors.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

print("Environment ready!")

In [ ]:
train = pd.read_csv("../train.csv")
test = pd.read_csv("../test.csv")
sample_submission = pd.read_csv("../sample_submission.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Sample submission:", sample_submission.shape)

In [ ]:
train.head(10)

In [ ]:
train.info()

In [ ]:
train["label"].describe()

In [ ]:
train["label"].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(train["label"], bins=20)

plt.xlabel("Grammar Score")
plt.ylabel("Number of Samples")
plt.title("Training Grammar Score Distribution")

plt.show()

In [ ]:
train_files = set(os.listdir("../train"))
test_files = set(os.listdir("../test"))

missing_train = train[
    ~train["filename"].isin(train_files)
]

missing_test = test[
    ~test["filename"].isin(test_files)
]

print("Training WAV files:", len(train_files))
print("Test WAV files:", len(test_files))

print("\nMissing training audio:")
print(missing_train)

print("\nMissing test audio:")
print(missing_test)

In [ ]:
print(
    "Duplicate train filenames:",
    train["filename"].duplicated().sum()
)

print(
    "Duplicate test filenames:",
    test["filename"].duplicated().sum()
)

print(
    "Train/Test overlap:",
    len(
        set(train["filename"]) &
        set(test["filename"])
    )
)

In [ ]:
def inspect_audio(path):
    y, sr = librosa.load(path, sr=None, mono=False)

    if y.ndim == 1:
        channels = 1
        samples = len(y)
    else:
        channels = y.shape[0]
        samples = y.shape[-1]

    duration = samples / sr

    return {
        "sample_rate": sr,
        "channels": channels,
        "samples": samples,
        "duration": duration
    }

In [ ]:
filename = train.iloc[0]["filename"]

audio_path = os.path.join("../train", filename)

info = inspect_audio(audio_path)

print("Filename:", filename)

for key, value in info.items():
    print(f"{key}: {value}")

In [ ]:
from IPython.display import Audio

Audio(audio_path)

In [ ]:
y, sr = librosa.load(audio_path, sr=None)

plt.figure(figsize=(14, 4))

librosa.display.waveshow(y, sr=sr)

plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.title(filename)

plt.show()

## Key observations

The inspection cells establish the available sample counts, target distribution, filename integrity, audio availability, and representative recording metadata. Together, these checks confirm whether rows can be joined safely across CSV files, whether the target is imbalanced or sparse at some scores, and whether audio-derived features are feasible. These facts guide the later choice to preserve filename alignment and evaluate continuous predictions with RMSE and Pearson correlation.